In [ ]:
from pathlib import Path
import os
import pandas as pd


def _resolve_orion_paths(strategy_code: str):
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env   = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve() if sig_env else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break

        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME (recommended).")

        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")

    return final_path, out_dir


In [ ]:
# -*- coding: utf-8 -*-
"""MARKET - pre-open regime. Label every session by how SPY and QQQ stand when it opens.

Published to signals/market/overnight_regime.json and read by the bridge's OvernightRegimeService,
which hangs each label on the session the window ENDS in - no offset, because the window already
belongs to that session.

THE WINDOW, AND WHY IT IS ANCHORED ON 16:00 RATHER THAN 21:00
The operator asked for 21:00 -> 09:30. Measured on this same file, that window cannot be built from
these rows:
  * 21:00 -> 09:30 comes out at sigma 1.135% (SPY) / 1.571% (QQQ), while 16:00 -> 09:30, which
    CONTAINS it, comes out at 0.632% / 0.856%. A shorter window inside a longer one cannot be 1.8x
    wider - the extra spread is the evening print, not the price.
  * the evening book is ~91 shares/min on SPY and ~168 on QQQ, against ~74,900 and ~56,600 in the
    regular session: 0.1-0.3% of normal size.
  * averaging does not rescue it. A single print, the median of 20:45-21:30, the median or VWAP of
    20:00-22:00 and a trimmed mean of 19:00-23:00 (99% coverage) all still come out at 1.09-1.57%,
    so the whole evening quote series is unreliable here, not one bad minute.
  * only 65-66% of sessions have a usable 21:00 print at all.
So the window keeps the END the operator asked for - 09:30, the open - and moves the anchor to the
official 16:00 close, which is the one price in that stretch that is not a thin-book quote.

HOW IT IS MEASURED - one reading, no subtraction. Stack% is the move vs prev_close and prev_close
ROLLS AT 16:01, so Stack% at 09:30 IS the 16:00 -> 09:30 move, already computed.

THE BANDS ARE OURS, BECAUSE NO PUBLISHED STANDARD EXISTS. The practitioner sources count a "real
gap" from a flat 0.1% (about 80% of SPY sessions since 1993, mean absolute gap about 0.4%) and call
1% "big" (15-20% of sessions); the academic overnight-return literature sets no cutoff at all. This
file's own numbers land on top of those: |move| > 0.1% on 80.6% of SPY sessions, mean |move| 0.42%.
So k = 0.5 of each symbol's OWN sigma: SPY +/-0.316%, QQQ +/-0.428%, which sits between the 0.25%
and 0.5% cutoffs practitioners use, and leaves every bucket with 100+ sessions.

A session is UP when BOTH indices clear +k*sigma, DOWN when BOTH clear -k*sigma, FLAT when BOTH are
inside, MIXED when they disagree, UNKNOWN when either has no usable print. UNKNOWN is its own
answer and must never be folded into FLAT.

CAVEAT FOR PRE-CLASS STRATEGIES: this window ends at 09:30, so for a class that TRADES 21:00 ->
09:30 the label overlaps the trade's own window. For anything entering at or after the open
(OPG-*, MOR-LAUNCH, PULLBACK...) it is strictly known before entry. `labels_to_midnight` below is
the strictly-earlier alternative for the PRE case.
"""
import io
import json
import os

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

SYMBOLS = ("SPY", "QQQ")
K = 0.5

FINAL_PATH, OUT_DIR = _resolve_orion_paths("market")
OUT = OUT_DIR / "overnight_regime.json"
print("Using FINAL parquet:", FINAL_PATH)


def row_groups_for(pf, sym):
    """Row groups are disjoint by ticker and sorted, so statistics find the symbol without a scan."""
    ti = [i for i, n in enumerate(pf.schema.names) if n == "ticker"][0]
    out = []
    for g in range(pf.metadata.num_row_groups):
        st = pf.metadata.row_group(g).column(ti).statistics
        if st is None or st.min is None:
            continue
        if str(st.min) <= sym <= str(st.max):
            out.append(g)
    return out


def load(pf, sym):
    parts = []
    for g in row_groups_for(pf, sym):
        d = pf.read_row_group(g, columns=["ticker", "dt", "Stack%"]).to_pandas()
        d = d[d["ticker"] == sym]
        if len(d):
            parts.append(d)
    if not parts:
        return None
    d = pd.concat(parts, ignore_index=True)
    dt = pd.to_datetime(d["dt"], errors="coerce", utc=True)
    d = d.assign(day=dt.dt.strftime("%Y-%m-%d"), smin=dt.dt.hour * 60 + dt.dt.minute)
    d["Stack%"] = pd.to_numeric(d["Stack%"], errors="coerce")
    return d.dropna(subset=["day", "Stack%"])


def at_open(d):
    """-> Series indexed by SESSION date: Stack% at ~09:30, i.e. the 16:00 -> 09:30 move."""
    w = d[(d["smin"] >= 9 * 60 + 25) & (d["smin"] <= 9 * 60 + 35)].copy()
    w["dd"] = (w["smin"] - (9 * 60 + 30)).abs()
    return w.sort_values(["day", "dd"]).drop_duplicates("day", keep="first").set_index("day")["Stack%"]


def to_midnight(d):
    """The strictly-earlier alternative: 16:00 -> 00:00, re-keyed onto the session that FOLLOWS it,
    so both label sets in this file are indexed the same way - by the session they describe."""
    late = d[d["smin"] >= 23 * 60 + 45].copy()
    early = d[d["smin"] <= 15].copy()
    late["key"] = late["day"]
    early["key"] = (pd.to_datetime(early["day"]) - pd.Timedelta(days=1)).dt.strftime("%Y-%m-%d")
    late["dd"] = 24 * 60 - late["smin"]
    early["dd"] = early["smin"]
    m = pd.concat([late, early], ignore_index=True).sort_values(["key", "dd"])
    return m.drop_duplicates("key", keep="first").set_index("key")["Stack%"]


def classify(moves, sigmas, days):
    labels, counts = {}, {}
    for day in days:
        sides = []
        for sym in SYMBOLS:
            v = moves[sym].get(day)
            if v is None or not np.isfinite(v):
                sides = None
                break
            b = K * sigmas[sym]
            sides.append(1 if v > b else (-1 if v < -b else 0))
        if sides is None:
            lab = "UNKNOWN"
        elif all(s == 1 for s in sides):
            lab = "UP"
        elif all(s == -1 for s in sides):
            lab = "DOWN"
        elif all(s == 0 for s in sides):
            lab = "FLAT"
        else:
            lab = "MIXED"
        labels[day] = lab
        counts[lab] = counts.get(lab, 0) + 1
    return labels, counts


pf = pq.ParquetFile(str(FINAL_PATH))
raw = {}
for sym in SYMBOLS:
    d = load(pf, sym)
    if d is None:
        raise RuntimeError("%s has no rows in %s - the regime file cannot be built" % (sym, FINAL_PATH))
    raw[sym] = d

open_mv = {s: at_open(raw[s]) for s in SYMBOLS}
mid_mv = {s: to_midnight(raw[s]) for s in SYMBOLS}
# One sigma per symbol per window, from the whole file.
open_sig = {s: float(open_mv[s].std()) for s in SYMBOLS}
mid_sig = {s: float(mid_mv[s].std()) for s in SYMBOLS}
for s in SYMBOLS:
    print("%s: 16:00->09:30 on %d sessions, sigma %.4f%%, band +/-%.4f%%  |  16:00->00:00 on %d, sigma %.4f%%"
          % (s, len(open_mv[s]), open_sig[s], K * open_sig[s], len(mid_mv[s]), mid_sig[s]))

# The session axis is every day either index has an open print for.
sessions = sorted(set().union(*[set(m.index) for m in open_mv.values()]))
# The midnight set is keyed by the session that FOLLOWS the night, so re-key it onto the session
# list: the night before session i is the one keyed to session i-1.
prev = {sessions[i]: sessions[i - 1] for i in range(1, len(sessions))}
mid_by_session = {s: pd.Series({d: mid_mv[s].get(prev.get(d)) for d in sessions}, dtype=float) for s in SYMBOLS}

labels, counts = classify(open_mv, open_sig, sessions)
labels_mid, counts_mid = classify(mid_by_session, mid_sig, sessions)

doc = {
    "meta": {
        "window": "16:00 -> 09:30 NY (the operator asked for 21:00 -> 09:30; the END is the same, "
                  "the anchor moved to the official close because the 21:00 print is a thin-book "
                  "quote - see the notebook header for the measurements)",
        "symbols": list(SYMBOLS),
        "k_sigma": K,
        "sigma_pct": {s: round(open_sig[s], 6) for s in SYMBOLS},
        "band_pct": {s: round(K * open_sig[s], 6) for s in SYMBOLS},
        "rule": ("UP when BOTH clear +k*sigma, DOWN when BOTH clear -k*sigma, FLAT when BOTH are "
                 "inside the band, MIXED when they disagree, UNKNOWN when either has no usable "
                 "print. UNKNOWN is not FLAT."),
        "reference": ("Stack% at 09:30 read as-is: prev_close rolls at 16:01, so at the open it IS "
                      "the previous regular-session close and Stack% is already the 16:00 -> 09:30 "
                      "move. No subtraction."),
        "keyed_by": "the SESSION the window ends in - the label needs no offset on the consumer side",
        "alt_window": "16:00 -> 00:00, in labels_to_midnight, keyed the same way; use it for a class "
                      "that TRADES 21:00 -> 09:30, where the main label would overlap the trade's "
                      "own window",
        "alt_sigma_pct": {s: round(mid_sig[s], 6) for s in SYMBOLS},
        "days": len(sessions),
        "counts": counts,
        "counts_to_midnight": counts_mid,
        "source": os.path.basename(str(FINAL_PATH)),
    },
    "labels": labels,
    "labels_to_midnight": labels_mid,
}

OUT.parent.mkdir(parents=True, exist_ok=True)
io.open(str(OUT), "w", encoding="utf-8", newline="\n").write(json.dumps(doc, indent=1, sort_keys=False))
total = len(sessions)
print("\n%d sessions labelled -> %s" % (total, OUT))
print("   %-9s %-22s %s" % ("", "16:00->09:30 (main)", "16:00->00:00 (alt)"))
for lab in ("UP", "DOWN", "FLAT", "MIXED", "UNKNOWN"):
    n, nm = counts.get(lab, 0), counts_mid.get(lab, 0)
    print("   %-9s %4d  %5.1f%%           %4d  %5.1f%%"
          % (lab, n, 100.0 * n / max(total, 1), nm, 100.0 * nm / max(total, 1)))
